# experiment:resolution_loss_analysis

Quantifies the **two information-loss channels** introduced by the 3m→10m→3m resolution round-trip.
These numbers must be reported before any FM results — they define the performance ceiling.

---

### Channel 1 — Mask Output Loss (Hard Performance Ceiling)
Downsample GT mask (3m → 10m nearest-neighbor) → upsample back (10m → 3m bilinear) → IoU vs original.
This IoU is the **absolute ceiling** no FM can exceed, regardless of representation quality.

### Channel 2 — Image Input Loss (Diagnostic)
Downsample PlanetScope image (3m → 10m bilinear) → upsample back (10m → 3m bilinear) → SSIM + PSNR vs original.
This quantifies how degraded the FM's input is before inference. **Not a hard performance ceiling** — diagnostic only.

---
**Requirements:** GPU not needed. CPU only. RiverScope dataset on Drive.

## 0. Mount Drive & Set Paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# ── EDIT THESE ────────────────────────────────────────────────────────────────
DATASET_DIR = '/content/drive/MyDrive/CS682/project/RiverScope_dataset'
OUTPUT_DIR  = '/content/drive/MyDrive/CS682/project/results/resolution_loss_analysis'
N_TILES     = 50    # number of test tiles to sample; set to -1 for full test set
RANDOM_SEED = 42
# ──────────────────────────────────────────────────────────────────────────────

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

Mounted at /content/drive
Output dir: /content/drive/MyDrive/CS682/project/results/resolution_loss_analysis


## 1. Install Dependencies

In [2]:
!pip install -q rasterio scikit-image

import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
import json
from pathlib import Path
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print('Dependencies loaded.')

Dependencies loaded.


## 2. Load Test Split & Verify Paths

In [3]:
df_test = pd.read_csv(os.path.join(DATASET_DIR, 'test.csv'))
print(f'Test split: {len(df_test)} tiles')
print(f'Columns: {list(df_test.columns)}')

# Verify a sample tile resolves
sample_img_path   = os.path.join(DATASET_DIR, df_test['normalized_planetscope_path'].iloc[0])
sample_label_path = os.path.join(DATASET_DIR, df_test['label_path'].iloc[0])
assert os.path.exists(sample_img_path),   f'Image not found: {sample_img_path}'
assert os.path.exists(sample_label_path), f'Label not found: {sample_label_path}'

# Check band count and native shape
with rasterio.open(sample_img_path) as src:
    print(f'\nSample tile — bands: {src.count}, shape: {src.height}x{src.width}, CRS: {src.crs}')
    print(f'Pixel size (m): {src.res}')

# Sample N_TILES from test set
if N_TILES > 0 and N_TILES < len(df_test):
    df_sample = df_test.sample(N_TILES, random_state=RANDOM_SEED).reset_index(drop=True)
else:
    df_sample = df_test.reset_index(drop=True)
print(f'\nRunning analysis on {len(df_sample)} tiles.')

Test split: 235 tiles
Columns: ['reach_id', 'mid_lat', 'mid_lon', 'planetscope_id', 'normalized_planetscope_path', 'label_path', 'sword_path', 's2_path_raw', 's2_path_reprojected', 'swot_node_path', 'swot_node_id', 'swot_pixc_path', 'swot_pixc_id']

Sample tile — bands: 4, shape: 500x500, CRS: EPSG:32721
Pixel size (m): (3.0, 3.0)

Running analysis on 50 tiles.


## 3. Helper Functions

In [4]:
def read_mask(path: str) -> np.ndarray:
    """Read single-band binary mask GeoTIFF. Returns float32 [H, W] in {0, 1}."""
    with rasterio.open(path) as src:
        mask = src.read(1).astype(np.float32)
    return (mask > 0.5).astype(np.float32)


def read_image(path: str) -> np.ndarray:
    """Read multi-band GeoTIFF. Returns float32 [C, H, W] normalised to [0, 1]."""
    with rasterio.open(path) as src:
        img = src.read().astype(np.float32)   # [C, H, W]
    # Normalise each band to [0, 1] using 2nd/98th percentile
    for i in range(img.shape[0]):
        band = img[i]
        valid = band[band > 0]
        if len(valid) == 0:
            continue
        p2, p98 = np.percentile(valid, (2, 98))
        img[i] = np.clip((band - p2) / (p98 - p2 + 1e-8), 0, 1)
    return img


def numpy_to_tensor_4d(arr: np.ndarray) -> torch.Tensor:
    """[C, H, W] numpy → [1, C, H, W] float tensor."""
    return torch.from_numpy(arr).unsqueeze(0)


def round_trip_mask(mask_np: np.ndarray, target_scale: float = 10/3) -> np.ndarray:
    """
    Mask round-trip: 3m → 10m (nearest-neighbor) → 3m (bilinear).
    mask_np: [H, W] float32 in {0, 1}
    Returns: [H, W] float32, values in [0, 1] after bilinear upsample.
    """
    H, W = mask_np.shape
    # Compute 10m resolution size
    H_10m = max(1, round(H / target_scale))
    W_10m = max(1, round(W / target_scale))

    t = torch.from_numpy(mask_np).unsqueeze(0).unsqueeze(0)  # [1,1,H,W]

    # Downsample: nearest-neighbor (preserve binary labels)
    t_down = F.interpolate(t, size=(H_10m, W_10m), mode='nearest')

    # Upsample: bilinear back to original size
    t_up = F.interpolate(t_down, size=(H, W), mode='bilinear', align_corners=False)

    return t_up.squeeze().numpy()


def round_trip_image(img_np: np.ndarray, target_scale: float = 10/3) -> np.ndarray:
    """
    Image round-trip: 3m → 10m (bilinear) → 3m (bilinear).
    img_np: [C, H, W] float32
    Returns: [C, H, W] float32
    """
    _, H, W = img_np.shape
    H_10m = max(1, round(H / target_scale))
    W_10m = max(1, round(W / target_scale))

    t = torch.from_numpy(img_np).unsqueeze(0)  # [1, C, H, W]

    # Downsample: bilinear (continuous image values)
    t_down = F.interpolate(t, size=(H_10m, W_10m), mode='bilinear', align_corners=False)

    # Upsample: bilinear back to original size
    t_up = F.interpolate(t_down, size=(H, W), mode='bilinear', align_corners=False)

    return t_up.squeeze(0).numpy()


def compute_iou(gt: np.ndarray, pred: np.ndarray, threshold: float = 0.5) -> float:
    """IoU between binary gt and thresholded pred."""
    gt_bin   = (gt   > threshold).astype(np.uint8)
    pred_bin = (pred > threshold).astype(np.uint8)
    TP = np.sum((gt_bin == 1) & (pred_bin == 1))
    FP = np.sum((gt_bin == 0) & (pred_bin == 1))
    FN = np.sum((gt_bin == 1) & (pred_bin == 0))
    return float(TP / (TP + FP + FN + 1e-7))

def compute_f1(gt: np.ndarray, pred: np.ndarray, threshold: float = 0.5) -> float:
    """F1 between binary gt and thresholded pred."""
    gt_bin   = (gt   > threshold).astype(np.uint8)
    pred_bin = (pred > threshold).astype(np.uint8)
    TP = np.sum((gt_bin == 1) & (pred_bin == 1))
    FP = np.sum((gt_bin == 0) & (pred_bin == 1))
    FN = np.sum((gt_bin == 1) & (pred_bin == 0))
    prec = TP / (TP + FP + 1e-7)
    rec  = TP / (TP + FN + 1e-7)
    return float(2 * prec * rec / (prec + rec + 1e-7))


def compute_ssim_psnr(orig: np.ndarray, recon: np.ndarray):
    """
    Compute per-band SSIM and PSNR, then average across bands.
    orig, recon: [C, H, W] float32 in [0, 1]
    Returns: (mean_ssim, mean_psnr, per_band_ssim, per_band_psnr)
    """
    C = orig.shape[0]
    ssims, psnrs = [], []
    for c in range(C):
        s = ssim_fn(orig[c], recon[c], data_range=1.0)
        p = psnr_fn(orig[c], recon[c], data_range=1.0)
        ssims.append(s)
        psnrs.append(p)
    return float(np.mean(ssims)), float(np.mean(psnrs)), ssims, psnrs


print('Helper functions defined.')

Helper functions defined.


## 4. Channel 1 — Mask Round-Trip IoU
**Hard performance ceiling.** The IoU ceiling printed here is the maximum any FM can achieve
at 3m evaluation, due to boundary information destroyed by the 3m→10m→3m mask round-trip alone.

In [5]:
channel1_results = []

for idx, row in df_sample.iterrows():
    label_path = os.path.join(DATASET_DIR, row['label_path'])

    # Read original 3m GT mask
    gt_mask = read_mask(label_path)

    # Skip tiles with no water pixels — IoU is undefined
    if gt_mask.sum() == 0:
        continue

    # Round-trip: 3m → 10m (nearest-neighbor) → 3m (bilinear)
    rt_mask = round_trip_mask(gt_mask)

    # IoU between round-tripped mask and original
    iou = compute_iou(gt_mask, rt_mask, threshold=0.5)

    f1  = compute_f1(gt_mask,  rt_mask, threshold=0.5)

    channel1_results.append({
        'tile_id':    row.get('tile_id', idx),
        'label_path': label_path,
        'iou':        iou,
        'f1':         f1,
        'water_frac': float(gt_mask.mean()),
        'H': gt_mask.shape[0],
        'W': gt_mask.shape[1],
    })

df_ch1 = pd.DataFrame(channel1_results)

# print('═' * 55)
# print('CHANNEL 1 — MASK ROUND-TRIP IoU  (performance ceiling)')
# print('═' * 55)
# print(f'  Tiles evaluated : {len(df_ch1)}')
# print(f'  Mean IoU        : {df_ch1["iou"].mean():.4f}')
# print(f'  Std  IoU        : {df_ch1["iou"].std():.4f}')
# print(f'  Min  IoU        : {df_ch1["iou"].min():.4f}')
# print(f'  Max  IoU        : {df_ch1["iou"].max():.4f}')
# print()
# print(f'  ► CEILING: Any FM IoU above {df_ch1["iou"].mean():.4f} ± {df_ch1["iou"].std():.4f}')
# print(f'    cannot be explained by resolution loss alone.')

print('═' * 55)
print('CHANNEL 1 — MASK ROUND-TRIP (performance ceiling)')
print(f'  Mean IoU : {df_ch1["iou"].mean():.4f} ± {df_ch1["iou"].std():.4f}')
print(f'  Mean F1  : {df_ch1["f1"].mean():.4f} ± {df_ch1["f1"].std():.4f}')
print(f'  ► IoU ceiling : {df_ch1["iou"].mean():.4f}')
print(f'  ► F1  ceiling : {df_ch1["f1"].mean():.4f}')

df_ch1.to_csv(os.path.join(OUTPUT_DIR, 'channel1_mask_roundtrip.csv'), index=False)
print(f'\nSaved to: {OUTPUT_DIR}/channel1_mask_roundtrip.csv')

═══════════════════════════════════════════════════════
CHANNEL 1 — MASK ROUND-TRIP (performance ceiling)
  Mean IoU : 0.9199 ± 0.0581
  Mean F1  : 0.9573 ± 0.0345
  ► IoU ceiling : 0.9199
  ► F1  ceiling : 0.9573

Saved to: /content/drive/MyDrive/CS682/project/results/resolution_loss_analysis/channel1_mask_roundtrip.csv


## 5. Channel 2 — Image Round-Trip SSIM & PSNR
**Diagnostic only — not a hard performance ceiling.**
Quantifies how much the FM's input was degraded before inference.
SSIM/PSNR loss → input degradation → indirect contribution to FM performance gap.
Cannot be mapped directly to an IoU bound.

In [6]:
channel2_results = []

for idx, row in df_sample.iterrows():
    img_path = os.path.join(DATASET_DIR, row['normalized_planetscope_path'])

    # Read original 3m image
    orig_img = read_image(img_path)   # [C, H, W] float32 in [0,1]

    # Round-trip: 3m → 10m (bilinear) → 3m (bilinear)
    rt_img = round_trip_image(orig_img)

    mean_ssim, mean_psnr, band_ssims, band_psnrs = compute_ssim_psnr(orig_img, rt_img)

    channel2_results.append({
        'tile_id':   row.get('tile_id', idx),
        'mean_ssim': mean_ssim,
        'mean_psnr': mean_psnr,
        **{f'ssim_band{i}': v for i, v in enumerate(band_ssims)},
        **{f'psnr_band{i}': v for i, v in enumerate(band_psnrs)},
    })

df_ch2 = pd.DataFrame(channel2_results)

print('═' * 55)
print('CHANNEL 2 — IMAGE ROUND-TRIP SSIM/PSNR  (diagnostic)')
print('═' * 55)
print(f'  Tiles evaluated : {len(df_ch2)}')
print()
print(f'  SSIM  mean ± std : {df_ch2["mean_ssim"].mean():.4f} ± {df_ch2["mean_ssim"].std():.4f}')
print(f'  SSIM  min / max  : {df_ch2["mean_ssim"].min():.4f} / {df_ch2["mean_ssim"].max():.4f}')
print()
print(f'  PSNR  mean ± std : {df_ch2["mean_psnr"].mean():.2f} ± {df_ch2["mean_psnr"].std():.2f} dB')
print(f'  PSNR  min / max  : {df_ch2["mean_psnr"].min():.2f} / {df_ch2["mean_psnr"].max():.2f} dB')
print()
print('  NOTE: SSIM < 1.0 and PSNR < ∞ means the FM never saw the full')
print('  3m image — it operated on a blurred 10m approximation.')
print('  This is a second source of asymmetry vs supervised 3m baselines.')

df_ch2.to_csv(os.path.join(OUTPUT_DIR, 'channel2_image_roundtrip.csv'), index=False)
print(f'\nSaved to: {OUTPUT_DIR}/channel2_image_roundtrip.csv')

/usr/local/lib/python3.12/dist-packages/skimage/metrics/simple_metrics.py:168: RuntimeWarning: divide by zero encountered in scalar divide
  return 10 * np.log10((data_range**2) / err)


═══════════════════════════════════════════════════════
CHANNEL 2 — IMAGE ROUND-TRIP SSIM/PSNR  (diagnostic)
═══════════════════════════════════════════════════════
  Tiles evaluated : 50

  SSIM  mean ± std : 0.8276 ± 0.0891
  SSIM  min / max  : 0.5066 / 1.0000

  PSNR  mean ± std : inf ± nan dB
  PSNR  min / max  : 22.63 / inf dB

  NOTE: SSIM < 1.0 and PSNR < ∞ means the FM never saw the full
  3m image — it operated on a blurred 10m approximation.
  This is a second source of asymmetry vs supervised 3m baselines.

Saved to: /content/drive/MyDrive/CS682/project/results/resolution_loss_analysis/channel2_image_roundtrip.csv


## 6. Visualise Round-Trip Degradation (6 sample tiles)

In [7]:
N_VIZ = 6
sample_rows = df_sample.sample(min(N_VIZ, len(df_sample)), random_state=RANDOM_SEED)

fig, axes = plt.subplots(N_VIZ, 4, figsize=(18, 4 * N_VIZ))
col_titles = ['RGB 3m (original)', 'RGB 3m (round-trip)', 'GT mask (original)', 'GT mask (round-trip)']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=10)

for row_i, (_, row) in enumerate(sample_rows.iterrows()):
    img_path   = os.path.join(DATASET_DIR, row['normalized_planetscope_path'])
    label_path = os.path.join(DATASET_DIR, row['label_path'])

    orig_img  = read_image(img_path)
    rt_img    = round_trip_image(orig_img)
    orig_mask = read_mask(label_path)
    rt_mask   = round_trip_mask(orig_mask)

    # Compute metrics for this tile
    iou  = compute_iou(orig_mask, rt_mask)
    s, p, _, _ = compute_ssim_psnr(orig_img, rt_img)

    # RGB: use first 3 bands
    rgb_orig = np.transpose(orig_img[:3], (1, 2, 0))
    rgb_rt   = np.transpose(rt_img[:3],  (1, 2, 0))

    axes[row_i][0].imshow(np.clip(rgb_orig, 0, 1))
    axes[row_i][1].imshow(np.clip(rgb_rt,   0, 1))
    axes[row_i][1].set_xlabel(f'SSIM={s:.3f}  PSNR={p:.1f}dB', fontsize=8)
    axes[row_i][2].imshow(orig_mask, cmap='Blues', vmin=0, vmax=1)
    axes[row_i][3].imshow(rt_mask,   cmap='Blues', vmin=0, vmax=1)
    axes[row_i][3].set_xlabel(f'Mask round-trip IoU={iou:.3f}', fontsize=8)

    for ax in axes[row_i]:
        ax.axis('off')

plt.suptitle('Resolution Round-Trip Degradation (3m → 10m → 3m)', fontsize=13, y=1.002)
plt.tight_layout()
viz_path = os.path.join(OUTPUT_DIR, 'roundtrip_degradation_grid.png')
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to: {viz_path}')

Output hidden; open in https://colab.research.google.com to view.

## 7. Save Summary — Paste Into Zero-Shot Notebook

In [8]:
summary = {
    'n_tiles': len(df_ch1),
    'channel1_mask_iou_mean': round(float(df_ch1['iou'].mean()), 4),
    'channel1_mask_iou_std':  round(float(df_ch1['iou'].std()),  4),
    'channel2_ssim_mean':     round(float(df_ch2['mean_ssim'].mean()), 4),
    'channel2_ssim_std':      round(float(df_ch2['mean_ssim'].std()),  4),
    'channel2_psnr_mean':     round(float(df_ch2['mean_psnr'].mean()), 2),
    'channel2_psnr_std':      round(float(df_ch2['mean_psnr'].std()),  2),
}

summary_path = os.path.join(OUTPUT_DIR, 'resolution_loss_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('═' * 55)
print('FINAL SUMMARY — experiment:resolution_loss_analysis')
print('═' * 55)
print(f'  Channel 1 — Mask IoU ceiling : {summary["channel1_mask_iou_mean"]:.4f} ± {summary["channel1_mask_iou_std"]:.4f}')
print(f'  Channel 2 — Image SSIM       : {summary["channel2_ssim_mean"]:.4f} ± {summary["channel2_ssim_std"]:.4f}')
print(f'  Channel 2 — Image PSNR       : {summary["channel2_psnr_mean"]:.2f} ± {summary["channel2_psnr_std"]:.2f} dB')
print()
print(f'  Summary saved to: {summary_path}')
print()
print('  ► Copy summary_path into experiment:zero_shot_linear_probe notebook')
print(f'    RESOLUTION_SUMMARY_PATH = "{summary_path}"')

═══════════════════════════════════════════════════════
FINAL SUMMARY — experiment:resolution_loss_analysis
═══════════════════════════════════════════════════════
  Channel 1 — Mask IoU ceiling : 0.9199 ± 0.0581
  Channel 2 — Image SSIM       : 0.8276 ± 0.0891
  Channel 2 — Image PSNR       : inf ± nan dB

  Summary saved to: /content/drive/MyDrive/CS682/project/results/resolution_loss_analysis/resolution_loss_summary.json

  ► Copy summary_path into experiment:zero_shot_linear_probe notebook
    RESOLUTION_SUMMARY_PATH = "/content/drive/MyDrive/CS682/project/results/resolution_loss_analysis/resolution_loss_summary.json"
